# Compute Correlations with Genomics for HK1: REDS Index and REDS Recall
## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from collections import defaultdict
from itertools import combinations, product
import sys

import numpy as np
import pandas as pd
import scikit_posthocs
import scipy as sp

from hk1_r12ter_analysis.config import (
    GENE_ID,
    INTERIM_DATA_DIR,
    PROCESSED_DATA_DIR,
    logger,
)
from hk1_r12ter_analysis.io import (
    load_dataframe_from_file,
    make_filename,
    save_dataframe_to_file,
)

# Suppress logging if desired
logger.remove(1)
logger.add(sys.stderr, level="WARNING");

## Load REDS Index data

In [ ]:
sample_key = "INDEX DONOR ID"
group_key = None
source_type = "REDS-Index"
main_data_type = f"Genomics-{GENE_ID}"
other_data_type = "All_Data"
norm_str = "median-none-auto"

# Input data arguments
input_dirpath = PROCESSED_DATA_DIR / "REDS" / "Normalized" / norm_str
# Temp data arguments
temp_dirpath = INTERIM_DATA_DIR / "REDS" / "Correlations"
temp_dirpath.mkdir(parents=True, exist_ok=True)
# Output data arguments

output_dirpath = PROCESSED_DATA_DIR / "REDS" / "Correlations"
output_dirpath.mkdir(parents=True, exist_ok=True)

#### All data

In [ ]:
input_filename = make_filename(source_type, "Donor", "RBC", "All_Data")
df_all_data = load_dataframe_from_file(
    input_dirpath / input_filename,
    index_col=[key for key in [sample_key, group_key] if key],
    header=[0, 1],
)
df_all_data = df_all_data.convert_dtypes()
if group_key:
    # Swap group and samples in index
    df_all_data.index = df_all_data.index.swaplevel(0, 1)
    df_all_data.index = df_all_data.index.set_levels(
        df_all_data.index.levels[0].astype(str), level=0
    )
    groups = df_all_data.index.get_level_values(0).unique().tolist()

df_all_data = df_all_data.select_dtypes(include=np.number)
df_data_main = df_all_data.loc[
    :,
    df_all_data.columns.isin(
        **dict(values=[main_data_type], level=0 if isinstance(main_data_type, str) else None)
    ),
].copy()
if other_data_type == "All_Data":
    df_data_other = df_all_data.loc[
        :,
        ~df_all_data.columns.isin(
            **dict(values=[main_data_type], level=0 if isinstance(main_data_type, str) else None)
        ),
    ].copy()
else:
    df_data_other = df_all_data.loc[
        :,
        df_all_data.columns.isin(
            **dict(values=[other_data_type], level=0 if isinstance(other_data_type, str) else None)
        ),
    ].copy()
df_data_main

#### Compute correlations with genomics

In [ ]:
# Save to interim directory prevents the need to recompute
recompute = False
nan_policy = "omit"
correlation_statistic = "Spearman"
output_filename = make_filename(
    source_type,
    "_".join(
        [
            main_data_type if isinstance(main_data_type, str) else "_".join(main_data_type),
            other_data_type if isinstance(other_data_type, str) else "_".join(other_data_type),
        ]
    ),
    correlation_statistic,
)

index_cols = ["Group", "DataType1", "Variable1", "DataType2", "Variable2"]
if not recompute:
    logger.debug("Loading previously computed correlations...")
    df_correlations = load_dataframe_from_file(
        temp_dirpath / output_filename, index_col=None, low_memory=False
    )
    df_correlations["Group"] = df_correlations["Group"].astype(str)
    logger.info("Loaded previously computed correlations.")
else:
    logger.info("Computing Spearman correlations...")
    correlations_dict = defaultdict(dict)

    for idx, (group, (col_names_main, series_main), (col_names_other, series_other)) in enumerate(
        product(
            ["ALL"] + (groups if group_key else []), df_data_main.items(), df_data_other.items()
        )
    ):
        if col_names_main == col_names_other:
            continue

        if group != "ALL":
            series_main = series_main.loc[group]
            series_other = series_other.loc[group]
        if len(series_other.unique()) == 1:
            continue
        result = sp.stats.spearmanr(
            series_main.values,
            series_other.values,
            axis=0,
            nan_policy=nan_policy,
            alternative="two-sided",
        )
        correlations_dict[idx]["Group"] = group
        correlations_dict[idx].update(dict(zip(["DataType1", "Variable1"], col_names_main)))
        correlations_dict[idx].update(dict(zip(["DataType2", "Variable2"], col_names_other)))
        correlations_dict[idx][f"{correlation_statistic} statistic"] = result.statistic
        correlations_dict[idx][f"{correlation_statistic} p-value"] = result.pvalue
        correlations_dict[idx][f"{correlation_statistic} -log10(p-value)"] = -np.log10(
            result.pvalue
        )
    df_correlations = pd.DataFrame.from_dict(correlations_dict, orient="index")
    logger.info("Computed Spearman correlations.")

    # Set groups as categories
    if df_correlations["Group"].size > 1:
        df_correlations["Group"] = pd.Categorical(df_correlations["Group"])
        df_correlations["Group"] = df_correlations["Group"].cat.reorder_categories(
            ["ALL"] + (groups if group_key else []), ordered=True
        )
    df_correlations = df_correlations.sort_values(
        ["Group", "DataType1", "DataType2", f"{correlation_statistic} -log10(p-value)"],
        ascending=[True, True, True, False],
    )
    df_correlations = df_correlations.reset_index(drop=True)
    save_dataframe_to_file(df_correlations, temp_dirpath / output_filename, index=False)

df_correlations = df_correlations.set_index(index_cols)
df_correlations

#### Stratify metadata values based on genotype allele count and test between groups

In [ ]:
# Save to interim directory prevents need to recompute
recompute = False
nan_policy = "omit"
alpha_normality = 0.05
alpha_homoscedasticity = 0.05
min_group_size = 2
always_use_nonparametric = False
output_filename = make_filename(
    source_type, "_".join([main_data_type, other_data_type]), "Statistics"
)

group_test_functions_dict = {"ANOVA": sp.stats.f_oneway, "Kruskal-Wallis": sp.stats.kruskal}
index_cols = ["Group", "DataType1", "Variable1", "DataType2", "Variable2"]
if not recompute:
    logger.debug("Loading previously computed correlations...")
    df_correlations_all = load_dataframe_from_file(
        output_dirpath / output_filename, index_col=index_cols, low_memory=False
    )
    logger.info("Loaded previously computed correlations.")
elif min_group_size < 2:
    raise ValueError("Minimum group size must be at least two data points per group")
else:
    stats_dict = defaultdict(dict)
    for group, datatype1, variable1, datatype2, variable2 in df_correlations.index:
        logger.info(f"Stratifying '{variable2}' data based on based on '{variable1}' groups.")
        df_var1_var2 = df_all_data.loc[:, [(datatype1, variable1), (datatype2, variable2)]]
        df_var1_var2 = df_var1_var2.droplevel(level=0, axis=1).dropna()
        subgroups = sorted(df_var1_var2[variable1].unique().astype(int))
        # Nothing to compare if only one genotype exists
        if len(subgroups) == 1:
            logger.info(
                f"Skipping '{variable1}' and '{variable2}' data, only one subgroup exists."
            )
            continue
        group_sizes = df_var1_var2.groupby(variable1).size()
        stats_dict[(group, datatype1, variable1, datatype2, variable2)].update(
            {f"Group size ({g:.0f})": s for g, s in group_sizes.items()}
        )
        if any(group_sizes < min_group_size):
            logger.info(
                f"Skipping '{variable1}' and '{variable2}' data, at least one subgroup is below the minimum sample size of {min_group_size}\n."
                + "; ".join(
                    [
                        f"Group '{g:.0f}' sample size: {s}"
                        for g, s in group_sizes[group_sizes < min_group_size].items()
                    ]
                )
            )
            continue
        # Normality of residuals test
        logger.debug("Testing normality...")
        residuals = df_var1_var2[variable2].sub(
            df_var1_var2.groupby(variable1)[variable2].transform("mean")
        )
        result = sp.stats.normaltest(residuals, axis=None, nan_policy=nan_policy)
        stats_dict[(group, datatype1, variable1, datatype2, variable2)][
            "Group Normality p-value"
        ] = result.pvalue
        assume_normality = result.pvalue > alpha_normality
        logger.info(
            "Normality can{assumption} be assumed (p-value = {pvalue:.3e} {sign} {alpha}).".format(
                assumption="" if assume_normality else "not",
                pvalue=result.pvalue,
                sign=">" if assume_normality else "<=",
                alpha=alpha_normality,
            )
        )
        # Homoscedasticity test
        logger.debug("Testing homoscedasticity...")
        df_subgroups = df_var1_var2.groupby(variable1)[variable2].agg(list)
        grouped_data = [df_subgroups[subgroup] for subgroup in subgroups]
        result = sp.stats.levene(*grouped_data, axis=0, nan_policy=nan_policy)
        stats_dict[(group, datatype1, variable1, datatype2, variable2)][
            "Group Homoscedasticity p-value"
        ] = result.pvalue
        assume_homoscedasticity = result.pvalue > alpha_homoscedasticity
        logger.info(
            "Homoscedasticity can{assumption} be assumed (p-value = {pvalue:.3e} {sign} {alpha}).".format(
                assumption="" if assume_homoscedasticity else "not",
                pvalue=result.pvalue,
                sign=">" if assume_homoscedasticity else "<=",
                alpha=alpha_homoscedasticity,
            )
        )
        # Test significance between groups
        assume_parametric = bool(
            assume_normality and assume_homoscedasticity and not always_use_nonparametric
        )
        group_test = "ANOVA" if assume_parametric else "Kruskal-Wallis"
        logger.debug(
            f"Performing '{group_test}' group test for '{variable1}' and '{variable2}' data."
        )
        group_test_function = group_test_functions_dict[group_test]
        result = group_test_function(*grouped_data, axis=0, nan_policy=nan_policy)
        stats_dict[(group, datatype1, variable1, datatype2, variable2)]["Group Test"] = group_test
        stats_dict[(group, datatype1, variable1, datatype2, variable2)][
            "Group p-value"
        ] = result.pvalue
        logger.info(
            f"Performed '{group_test}' group test for '{variable1}' and '{variable2}' data (p-value = {result.pvalue:.3e})."
        )

        if assume_parametric:
            group_combos = sorted([tuple(sorted(combo)) for combo in combinations(subgroups, 2)])
            # Check normality of each pairwise combination to apply same test across pairwsie combinations
            logger.debug("Testing homoscedasticity for all pairwise combinations of groups...")
            norms = {
                tuple(combo): sp.stats.levene(
                    *[df_subgroups[c] for c in combo], axis=0, nan_policy=nan_policy
                ).pvalue
                for combo in group_combos
            }
            # All pairwise combos must demonstrate normality to apply equal variance assumption.
            # If one combo doesn't demonstrate normality, don't apply equal variance assumption to any combos.
            assume_homoscedasticity = all([x > alpha_homoscedasticity for x in norms.values()])
            logger.info(
                "Homoscedasticity can{assumption} be assumed for all pairwise combinations of groups{pvalue}.".format(
                    assumption="" if assume_homoscedasticity else "not",
                    pvalue=(
                        ""
                        if assume_homoscedasticity
                        else "".join(
                            (
                                "(",
                                ";".join(
                                    [
                                        f"Groups ({g1:.0f},{g2:.0f}): {pvalue} <= {alpha_homoscedasticity}"
                                        for ((g1, g2), pvalue) in norms.items()
                                    ]
                                ),
                                ")",
                            )
                        )
                    ),
                )
            )
            # Test all pairwise combinations at once
            # Unpack results with element at index (i, j) is the p-value for the comparison between groups i and j
            pairwise_test = "Tukey-Kramer" if assume_homoscedasticity else "Games-Howell"
            logger.debug(
                f"Using the '{pairwise_test}' test for post hoc testing between groups..."
            )
            pairwise_pvalues = {
                "Pairwise p-value ({0:.0f},{1:.0f})".format(*combo): sp.stats.tukey_hsd(
                    *[df_subgroups[c] for c in combo], equal_var=assume_homoscedasticity
                ).pvalue[0, 1]
                for combo in group_combos
            }
        else:
            # Test all pairwise combinations at once
            pairwise_test = "Dunn"
            logger.debug(
                f"Using the '{pairwise_test}' test for post hoc testing between groups..."
            )
            result = scikit_posthocs.posthoc_dunn(
                df_var1_var2, val_col=variable2, group_col=variable1, p_adjust=None
            )
            pairwise_pvalues = {
                "Pairwise p-value ({0:.0f},{1:.0f})".format(g1, g2): value
                for g1, row in result.iterrows()
                for g2, value in row.items()
                if g1 < g2
            }

        stats_dict[(group, datatype1, variable1, datatype2, variable2)][
            "Pairwise Test"
        ] = pairwise_test
        stats_dict[(group, datatype1, variable1, datatype2, variable2)].update(pairwise_pvalues)
        logger.debug(f"Utilized the '{pairwise_test}' test for post hoc testing between groups.")
        logger.info(
            f"Finished testing '{variable2}' data after stratification into groups by '{variable1}'."
        )
    # Convert dictionary into DataFrame and merge with other correlation calculations
    df_statistics = pd.DataFrame.from_dict(stats_dict, orient="index")
    df_statistics.index.names = index_cols
    logger.debug("Merging statistical testing results into one DataFrame...")
    df_correlations_all = df_correlations.join(df_statistics, how="left")
    df_correlations_all = df_correlations_all.convert_dtypes()
    save_dataframe_to_file(df_correlations_all, output_dirpath / output_filename, index=True)
df_correlations_all

## Load REDS Recall data

In [ ]:
sample_key = "RECALL DONOR ID"
group_key = "Day"
source_type = "REDS-Recall"
main_data_type = f"Genomics-{GENE_ID}"
other_data_type = "All_Data"
norm_str = "median-none-auto"

# Input data arguments
input_dirpath = PROCESSED_DATA_DIR / "REDS" / "Normalized" / norm_str
# Temp data arguments
temp_dirpath = INTERIM_DATA_DIR / "REDS" / "Correlations"
temp_dirpath.mkdir(parents=True, exist_ok=True)
# Output data arguments
output_dirpath = PROCESSED_DATA_DIR / "REDS" / "Correlations"
output_dirpath.mkdir(parents=True, exist_ok=True)

#### All data

In [ ]:
input_filename = make_filename(source_type, "Donor", "RBC", "All_Data")
df_all_data = load_dataframe_from_file(
    input_dirpath / input_filename,
    index_col=[key for key in [sample_key, group_key] if key],
    header=[0, 1],
)
df_all_data = df_all_data.convert_dtypes()
if group_key:
    # Swap group and samples in index
    df_all_data.index = df_all_data.index.swaplevel(0, 1)
    df_all_data.index = df_all_data.index.set_levels(
        df_all_data.index.levels[0].astype(str), level=0
    )
    groups = df_all_data.index.get_level_values(0).unique().tolist()

df_all_data = df_all_data.select_dtypes(include=np.number)
df_data_main = df_all_data.loc[
    :,
    df_all_data.columns.isin(
        **dict(values=[main_data_type], level=0 if isinstance(main_data_type, str) else None)
    ),
].copy()
if other_data_type == "All_Data":
    df_data_other = df_all_data.loc[
        :,
        ~df_all_data.columns.isin(
            **dict(values=[main_data_type], level=0 if isinstance(main_data_type, str) else None)
        ),
    ].copy()
else:
    df_data_other = df_all_data.loc[
        :,
        df_all_data.columns.isin(
            **dict(values=[other_data_type], level=0 if isinstance(other_data_type, str) else None)
        ),
    ].copy()
df_data_main

#### Compute correlations with genomics

In [ ]:
# Save to interim directory prevents the need to recompute
recompute = False
nan_policy = "omit"
correlation_statistic = "Spearman"
output_filename = make_filename(
    source_type,
    "_".join(
        [
            main_data_type if isinstance(main_data_type, str) else "_".join(main_data_type),
            other_data_type if isinstance(other_data_type, str) else "_".join(other_data_type),
        ]
    ),
    correlation_statistic,
)

index_cols = ["Group", "DataType1", "Variable1", "DataType2", "Variable2"]
if not recompute:
    logger.debug("Loading previously computed correlations...")
    df_correlations = load_dataframe_from_file(
        temp_dirpath / output_filename, index_col=None, low_memory=False
    )
    df_correlations["Group"] = df_correlations["Group"].astype(str)
    logger.info("Loaded previously computed correlations.")
else:
    logger.info("Computing Spearman correlations...")
    correlations_dict = defaultdict(dict)

    for idx, (group, (col_names_main, series_main), (col_names_other, series_other)) in enumerate(
        product(
            ["ALL"] + (groups if group_key else []), df_data_main.items(), df_data_other.items()
        )
    ):
        if col_names_main == col_names_other:
            continue

        if group != "ALL":
            series_main = series_main.loc[group]
            series_other = series_other.loc[group]
        if len(series_other.unique()) == 1:
            continue
        result = sp.stats.spearmanr(
            series_main.values,
            series_other.values,
            axis=0,
            nan_policy=nan_policy,
            alternative="two-sided",
        )
        correlations_dict[idx]["Group"] = group
        correlations_dict[idx].update(dict(zip(["DataType1", "Variable1"], col_names_main)))
        correlations_dict[idx].update(dict(zip(["DataType2", "Variable2"], col_names_other)))
        correlations_dict[idx][f"{correlation_statistic} statistic"] = result.statistic
        correlations_dict[idx][f"{correlation_statistic} p-value"] = result.pvalue
        correlations_dict[idx][f"{correlation_statistic} -log10(p-value)"] = -np.log10(
            result.pvalue
        )
    df_correlations = pd.DataFrame.from_dict(correlations_dict, orient="index")
    logger.info("Computed Spearman correlations.")

    # Set groups as categories
    if df_correlations["Group"].size > 1:
        df_correlations["Group"] = pd.Categorical(df_correlations["Group"])
        df_correlations["Group"] = df_correlations["Group"].cat.reorder_categories(
            ["ALL"] + (groups if group_key else []), ordered=True
        )
    df_correlations = df_correlations.sort_values(
        ["Group", "DataType1", "DataType2", f"{correlation_statistic} -log10(p-value)"],
        ascending=[True, True, True, False],
    )
    df_correlations = df_correlations.reset_index(drop=True)
    save_dataframe_to_file(df_correlations, temp_dirpath / output_filename, index=False)

df_correlations = df_correlations.set_index(index_cols)
df_correlations

#### Stratify metadata values based on genotype allele count and test between groups

In [ ]:
# Save to interim directory prevents need to recompute
recompute = False
nan_policy = "omit"
alpha_normality = 0.05
alpha_homoscedasticity = 0.05
min_group_size = 2
always_use_nonparametric = False
output_filename = make_filename(
    source_type, "_".join([main_data_type, other_data_type]), "Statistics"
)

group_test_functions_dict = {"ANOVA": sp.stats.f_oneway, "Kruskal-Wallis": sp.stats.kruskal}
index_cols = ["Group", "DataType1", "Variable1", "DataType2", "Variable2"]
if not recompute:
    logger.debug("Loading previously computed correlations...")
    df_correlations_all = load_dataframe_from_file(
        output_dirpath / output_filename, index_col=index_cols, low_memory=False
    )
    logger.info("Loaded previously computed correlations.")
elif min_group_size < 2:
    raise ValueError("Minimum group size must be at least two data points per group")
else:
    stats_dict = defaultdict(dict)
    for group, datatype1, variable1, datatype2, variable2 in df_correlations.index:
        logger.info(f"Stratifying '{variable2}' data based on based on '{variable1}' groups.")
        df_var1_var2 = df_all_data.loc[:, [(datatype1, variable1), (datatype2, variable2)]]
        df_var1_var2 = df_var1_var2.droplevel(level=0, axis=1).dropna()
        subgroups = sorted(df_var1_var2[variable1].unique().astype(int))
        # Nothing to compare if only one genotype exists
        if len(subgroups) == 1:
            logger.info(
                f"Skipping '{variable1}' and '{variable2}' data, only one subgroup exists."
            )
            continue
        group_sizes = df_var1_var2.groupby(variable1).size()
        stats_dict[(group, datatype1, variable1, datatype2, variable2)].update(
            {f"Group size ({g:.0f})": s for g, s in group_sizes.items()}
        )
        if any(group_sizes < min_group_size):
            logger.info(
                f"Skipping '{variable1}' and '{variable2}' data, at least one subgroup is below the minimum sample size of {min_group_size}\n."
                + "; ".join(
                    [
                        f"Group '{g:.0f}' sample size: {s}"
                        for g, s in group_sizes[group_sizes < min_group_size].items()
                    ]
                )
            )
            continue
        # Normality of residuals test
        logger.debug("Testing normality...")
        residuals = df_var1_var2[variable2].sub(
            df_var1_var2.groupby(variable1)[variable2].transform("mean")
        )
        result = sp.stats.normaltest(residuals, axis=None, nan_policy=nan_policy)
        stats_dict[(group, datatype1, variable1, datatype2, variable2)][
            "Group Normality p-value"
        ] = result.pvalue
        assume_normality = result.pvalue > alpha_normality
        logger.info(
            "Normality can{assumption} be assumed (p-value = {pvalue:.3e} {sign} {alpha}).".format(
                assumption="" if assume_normality else "not",
                pvalue=result.pvalue,
                sign=">" if assume_normality else "<=",
                alpha=alpha_normality,
            )
        )
        # Homoscedasticity test
        logger.debug("Testing homoscedasticity...")
        df_subgroups = df_var1_var2.groupby(variable1)[variable2].agg(list)
        grouped_data = [df_subgroups[subgroup] for subgroup in subgroups]
        result = sp.stats.levene(*grouped_data, axis=0, nan_policy=nan_policy)
        stats_dict[(group, datatype1, variable1, datatype2, variable2)][
            "Group Homoscedasticity p-value"
        ] = result.pvalue
        assume_homoscedasticity = result.pvalue > alpha_homoscedasticity
        logger.info(
            "Homoscedasticity can{assumption} be assumed (p-value = {pvalue:.3e} {sign} {alpha}).".format(
                assumption="" if assume_homoscedasticity else "not",
                pvalue=result.pvalue,
                sign=">" if assume_homoscedasticity else "<=",
                alpha=alpha_homoscedasticity,
            )
        )
        # Test significance between groups
        assume_parametric = bool(
            assume_normality and assume_homoscedasticity and not always_use_nonparametric
        )
        group_test = "ANOVA" if assume_parametric else "Kruskal-Wallis"
        logger.debug(
            f"Performing '{group_test}' group test for '{variable1}' and '{variable2}' data."
        )
        group_test_function = group_test_functions_dict[group_test]
        result = group_test_function(*grouped_data, axis=0, nan_policy=nan_policy)
        stats_dict[(group, datatype1, variable1, datatype2, variable2)]["Group Test"] = group_test
        stats_dict[(group, datatype1, variable1, datatype2, variable2)][
            "Group p-value"
        ] = result.pvalue
        logger.info(
            f"Performed '{group_test}' group test for '{variable1}' and '{variable2}' data (p-value = {result.pvalue:.3e})."
        )

        if assume_parametric:
            group_combos = sorted([tuple(sorted(combo)) for combo in combinations(subgroups, 2)])
            # Check normality of each pairwise combination to apply same test across pairwsie combinations
            logger.debug("Testing homoscedasticity for all pairwise combinations of groups...")
            norms = {
                tuple(combo): sp.stats.levene(
                    *[df_subgroups[c] for c in combo], axis=0, nan_policy=nan_policy
                ).pvalue
                for combo in group_combos
            }
            # All pairwise combos must demonstrate normality to apply equal variance assumption.
            # If one combo doesn't demonstrate normality, don't apply equal variance assumption to any combos.
            assume_homoscedasticity = all([x > alpha_homoscedasticity for x in norms.values()])
            logger.info(
                "Homoscedasticity can{assumption} be assumed for all pairwise combinations of groups{pvalue}.".format(
                    assumption="" if assume_homoscedasticity else "not",
                    pvalue=(
                        ""
                        if assume_homoscedasticity
                        else "".join(
                            (
                                "(",
                                ";".join(
                                    [
                                        f"Groups ({g1:.0f},{g2:.0f}): {pvalue} <= {alpha_homoscedasticity}"
                                        for ((g1, g2), pvalue) in norms.items()
                                    ]
                                ),
                                ")",
                            )
                        )
                    ),
                )
            )
            # Test all pairwise combinations at once
            # Unpack results with element at index (i, j) is the p-value for the comparison between groups i and j
            pairwise_test = "Tukey-Kramer" if assume_homoscedasticity else "Games-Howell"
            logger.debug(
                f"Using the '{pairwise_test}' test for post hoc testing between groups..."
            )
            pairwise_pvalues = {
                "Pairwise p-value ({0:.0f},{1:.0f})".format(*combo): sp.stats.tukey_hsd(
                    *[df_subgroups[c] for c in combo], equal_var=assume_homoscedasticity
                ).pvalue[0, 1]
                for combo in group_combos
            }
        else:
            # Test all pairwise combinations at once
            pairwise_test = "Dunn"
            logger.debug(
                f"Using the '{pairwise_test}' test for post hoc testing between groups..."
            )
            result = scikit_posthocs.posthoc_dunn(
                df_var1_var2, val_col=variable2, group_col=variable1, p_adjust=None
            )
            pairwise_pvalues = {
                "Pairwise p-value ({0:.0f},{1:.0f})".format(g1, g2): value
                for g1, row in result.iterrows()
                for g2, value in row.items()
                if g1 < g2
            }

        stats_dict[(group, datatype1, variable1, datatype2, variable2)][
            "Pairwise Test"
        ] = pairwise_test
        stats_dict[(group, datatype1, variable1, datatype2, variable2)].update(pairwise_pvalues)
        logger.debug(f"Utilized the '{pairwise_test}' test for post hoc testing between groups.")
        logger.info(
            f"Finished testing '{variable2}' data after stratification into groups by '{variable1}'."
        )
    # Convert dictionary into DataFrame and merge with other correlation calculations
    df_statistics = pd.DataFrame.from_dict(stats_dict, orient="index")
    df_statistics.index.names = index_cols
    logger.debug("Merging statistical testing results into one DataFrame...")
    df_correlations_all = df_correlations.join(df_statistics, how="left")
    df_correlations_all = df_correlations_all.convert_dtypes()
    save_dataframe_to_file(df_correlations_all, output_dirpath / output_filename, index=True)
df_correlations_all